# Benchmarks DTLZ / MaF — NSGA-III vs QI-NSGA-III

Ce notebook reproduit, sur les suites de test academiques standard **DTLZ1-7**
et **MaF1-7** (Cheng et al. 2017), la validation deja effectuee dans le projet
pour comparer **NSGA-III** et **QI-NSGA-III** sur des problemes de reference
(en complement de la validation sur l'IRP metier, qui fait l'objet d'un autre
notebook). Protocole : Cui, Shi, Wang, Ding, Li & Li (2025), *Complex &
Intelligent Systems* 11:136 — SBX eta=20 pc=1.0, PM eta=20 pm=1/D, directions
de reference Das-Dennis, budget fixe de 30000 evaluations par run.

**Tout est auto-contenu** : aucun fichier externe a uploader.

**Marche a suivre : Exécution -> Tout exécuter** (ou Ctrl+F9), puis attendre.

Avec les reglages par defaut ci-dessous (`N_RUNS=5`, M=3 et M=4, DTLZ et MaF),
le calcul prend environ **45-90 minutes** selon les CPU alloues par Colab.
Pour un premier apercu rapide, reduisez `N_RUNS` (ex: 2) et/ou `SUITES`/
`N_OBJ_LIST` dans la derniere cellule avant de lancer le calcul complet.
Pour reproduire exactement le protocole de l'etude (30 runs), mettez
`N_RUNS = 30` (plusieurs heures).

A la fin : un tableau **NSGA-III vs QI-NSGA-III** (IGD moyen, qui gagne) par
probleme s'affiche, et une archive **.zip** de tous les CSV de resultats est
**telechargee automatiquement** dans votre navigateur.

In [ ]:
!pip -q install pymoo

### 1. Imports

In [ ]:
from __future__ import annotations

import argparse
import csv
import os
import shutil
import sys
import time

import numpy as np

from pymoo.algorithms.moo.nsga3 import ReferenceDirectionSurvival
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.core.population import Population
from pymoo.core.problem import Problem
from pymoo.indicators.igd import IGD
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
from pymoo.problems import get_problem as _pymoo_get_problem
from pymoo.termination import get_termination
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.util.ref_dirs import get_reference_directions


### 2. Seeds partagees (reproductibilite)

In [ ]:
# =============================================================================
# 1. SEEDS PARTAGEES -- meme graine = meme condition initiale pour les 2 algos
# =============================================================================

SEEDS = [
    42, 137, 271, 491, 613, 733, 857, 977, 1009, 1123,
    1249, 1373, 1499, 1609, 1733, 1871, 1997, 2113, 2237, 2351,
    2467, 2593, 2711, 2837, 2953, 3079, 3191, 3313, 3433, 3557,
]


### 3. Configuration commune (directions de reference Das-Dennis + taille de population)

In [ ]:
# =============================================================================
# 2. CONFIGURATION COMMUNE -- directions de reference Das-Dennis + taille pop
#    (Cui et al. 2025, Table 2 : M=3 -> p=12 -> H=91 -> N=92 ; M=4 -> p=7 ->
#    H=120 -> N=120. Seuls M=3 et M=4 sont couverts par ce protocole.)
# =============================================================================

_N_OBJ_TO_P = {3: 12, 4: 7}


def get_run_config(n_obj: int):
    """Retourne (ref_dirs, pop_size) pour un nombre d'objectifs donne.

    pop_size = plus petit multiple de 4 >= nombre de directions de reference.
    """
    if n_obj not in _N_OBJ_TO_P:
        raise ValueError(
            f"n_obj={n_obj} non supporte. Valeurs supportees : {sorted(_N_OBJ_TO_P)}"
        )
    p = _N_OBJ_TO_P[n_obj]
    ref_dirs = get_reference_directions("das-dennis", n_obj, n_partitions=p)
    h = len(ref_dirs)
    pop_size = h + (4 - h % 4) % 4
    return ref_dirs, pop_size


TMAX = 30000  # budget fixe d'evaluations par run (Cui et al. 2025, Table 2)


### 4. Problemes DTLZ1-7 (via pymoo)

In [ ]:
# =============================================================================
# 3. PROBLEMES DTLZ1-7 (via pymoo)
#
#    n_var = n_obj + k - 1 (convention DTLZ standard, Deb et al. 2002) :
#      DTLZ1: k=5 ; DTLZ2-6: k=10 ; DTLZ7: k=20
#
#    DTLZ5/6/7 : front de Pareto degenere/disconnecte -- pymoo ne fournit un
#    front de reference precalcule qu'a M=3 pour ces trois-la (IGD non
#    calculable a M=4 ici).
# =============================================================================

DTLZ_PROBLEM_NAMES = ["DTLZ1", "DTLZ2", "DTLZ3", "DTLZ4", "DTLZ5", "DTLZ6", "DTLZ7"]
DTLZ_DEGENERATE_PROBLEMS = {"DTLZ5", "DTLZ6", "DTLZ7"}

_DTLZ_K = {
    "DTLZ1": 5, "DTLZ2": 10, "DTLZ3": 10, "DTLZ4": 10, "DTLZ5": 10,
    "DTLZ6": 10, "DTLZ7": 20,
}


def get_dtlz_problem(name: str, n_obj: int = 4):
    """Retourne (probleme pymoo, n_generations) pour un nom DTLZ1..DTLZ7."""
    name = name.upper()
    if name not in _DTLZ_K:
        raise ValueError(f"Probleme inconnu '{name}'. Choix : {DTLZ_PROBLEM_NAMES}")
    n_var = n_obj + _DTLZ_K[name] - 1
    problem = _pymoo_get_problem(name.lower(), n_var=n_var, n_obj=n_obj)
    _, pop_size = get_run_config(n_obj)
    n_gen = TMAX // pop_size
    return problem, n_gen


### 5. Problemes MaF1-7 (implementation directe, PlatEMO -> Python)

In [ ]:
# =============================================================================
# 4. PROBLEMES MaF1-7
#    Cheng, Li, Tian, Zhang, Yang, Jin & Yao (2017), "A benchmark test suite
#    for evolutionary many-objective optimization", Complex & Intelligent
#    Systems 3(1), 67-81. Traduction directe de l'implementation MATLAB de
#    reference (PlatEMO, BIMK/PlatEMO Problems/.../MaF/MaF{1..7}.m).
#
#    n_var = n_obj + k - 1, k=10 (MaF1-6) ou k=20 (MaF7, comme DTLZ7).
# =============================================================================

_MAF_K = {
    "MAF1": 10, "MAF2": 10, "MAF3": 10, "MAF4": 10, "MAF5": 10, "MAF6": 10,
    "MAF7": 20,
}


def _uniform_simplex_points(n_obj, n_partitions=12):
    """Points sur le simplexe unite (somme=1), i.e. PlatEMO's UniformPoint(N,M)."""
    return get_reference_directions("das-dennis", n_obj, n_partitions=n_partitions)


class MaF1(Problem):
    """DTLZ1 inverse -- front lineaire, oriente a l'oppose de l'origine."""

    def __init__(self, n_var, n_obj):
        super().__init__(n_var=n_var, n_obj=n_obj, xl=0.0, xu=1.0, vtype=float)

    def _evaluate(self, x, out, *args, **kwargs):
        M = self.n_obj
        n = x.shape[0]
        X_, X_M = x[:, :M - 1], x[:, M - 1:]
        g = np.sum((X_M - 0.5) ** 2, axis=1)
        ones_col = np.ones((n, 1))
        cp = np.fliplr(np.cumprod(np.hstack([ones_col, X_]), axis=1))
        rev = np.hstack([ones_col, 1 - X_[:, ::-1]])
        factor = cp * rev
        out["F"] = (1 + g)[:, None] - (1 + g)[:, None] * factor

    def _calc_pareto_front(self, ref_dirs=None):
        if ref_dirs is None:
            ref_dirs = _uniform_simplex_points(self.n_obj)
        return 1 - ref_dirs


class MaF2(Problem):
    """DTLZ2BZ -- front spherique restreint a une bande angulaire etroite."""

    def __init__(self, n_var, n_obj):
        super().__init__(n_var=n_var, n_obj=n_obj, xl=0.0, xu=1.0, vtype=float)

    def _group_slices(self):
        M, D = self.n_obj, self.n_var
        k = D - M + 1
        nk = k // M
        for m in range(M):
            if m < M - 1:
                yield slice(m * nk, (m + 1) * nk)
            else:
                yield slice((M - 1) * nk, k)

    def _evaluate(self, x, out, *args, **kwargs):
        M = self.n_obj
        n = x.shape[0]
        X_, X_M = x[:, :M - 1], x[:, M - 1:]
        g = np.zeros((n, M))
        for m, seg in enumerate(self._group_slices()):
            g[:, m] = np.sum((X_M[:, seg] / 2 + 0.25 - 0.5) ** 2, axis=1)
        theta = X_ / 2 + 0.25
        ones_col = np.ones((n, 1))
        cp = np.fliplr(np.cumprod(np.hstack([ones_col, np.cos(theta * np.pi / 2)]), axis=1))
        rev = np.hstack([ones_col, np.sin(theta[:, ::-1] * np.pi / 2)])
        out["F"] = (1 + g) * cp * rev

    @staticmethod
    def _reconstruct_c(R):
        """Reconstruit cos(theta) a partir des points du simplexe R
        (boucle recursive de PlatEMO MaF2.GetOptimum, traduction directe)."""
        n, M = R.shape
        c = np.zeros((n, M - 1))
        with np.errstate(divide="ignore", invalid="ignore"):
            for j in range(2, M + 1):
                t = M - j
                prod_term = np.prod(c[:, t + 1:M - 1], axis=1)
                temp = R[:, j - 1] / R[:, 0] * prod_term
                c[:, t] = np.sqrt(1.0 / (1.0 + temp ** 2))
        return c

    _OVERSAMPLE_PARTITIONS = {3: 324, 4: 89}

    def _calc_pareto_front(self, ref_dirs=None):
        p = self._OVERSAMPLE_PARTITIONS.get(self.n_obj, 34)
        R = _uniform_simplex_points(self.n_obj, n_partitions=p)
        c = self._reconstruct_c(R)
        lo, hi = np.cos(3 * np.pi / 8), np.cos(np.pi / 8)
        valid = np.all((c >= lo) & (c <= hi), axis=1)
        c = c[valid]
        n = c.shape[0]
        ones_col = np.ones((n, 1))
        cp = np.fliplr(np.cumprod(np.hstack([ones_col, c]), axis=1))
        rev = np.hstack([ones_col, np.sqrt(1 - c[:, ::-1] ** 2)])
        return cp * rev


class MaF3(Problem):
    """DTLZ3 convexe -- fonction de distance multimodale, front convexe."""

    def __init__(self, n_var, n_obj):
        super().__init__(n_var=n_var, n_obj=n_obj, xl=0.0, xu=1.0, vtype=float)

    def _evaluate(self, x, out, *args, **kwargs):
        M, D = self.n_obj, self.n_var
        n = x.shape[0]
        X_, X_M = x[:, :M - 1], x[:, M - 1:]
        k = D - M + 1
        g = 100 * (k + np.sum((X_M - 0.5) ** 2 - np.cos(20 * np.pi * (X_M - 0.5)), axis=1))
        ones_col = np.ones((n, 1))
        cp = np.fliplr(np.cumprod(np.hstack([ones_col, np.cos(X_ * np.pi / 2)]), axis=1))
        rev = np.hstack([ones_col, np.sin(X_[:, ::-1] * np.pi / 2)])
        f = (1 + g)[:, None] * cp * rev
        out["F"] = np.hstack([f[:, :M - 1] ** 4, f[:, M - 1:M] ** 2])

    def _calc_pareto_front(self, ref_dirs=None):
        if ref_dirs is None:
            ref_dirs = _uniform_simplex_points(self.n_obj)
        R = ref_dirs ** 2
        temp = np.sum(np.sqrt(R[:, :-1]), axis=1) + R[:, -1]
        R = R.copy()
        R[:, :-1] = R[:, :-1] / (temp ** 2)[:, None]
        R[:, -1] = R[:, -1] / temp
        return R


class MaF4(Problem):
    """DTLZ3 inverse et mise a l'echelle 2^1..2^M -- meme g multimodal que MaF3."""

    def __init__(self, n_var, n_obj):
        super().__init__(n_var=n_var, n_obj=n_obj, xl=0.0, xu=1.0, vtype=float)

    def _evaluate(self, x, out, *args, **kwargs):
        M, D = self.n_obj, self.n_var
        n = x.shape[0]
        X_, X_M = x[:, :M - 1], x[:, M - 1:]
        k = D - M + 1
        g = 100 * (k + np.sum((X_M - 0.5) ** 2 - np.cos(20 * np.pi * (X_M - 0.5)), axis=1))
        ones_col = np.ones((n, 1))
        cp = np.fliplr(np.cumprod(np.hstack([ones_col, np.cos(X_ * np.pi / 2)]), axis=1))
        rev = np.hstack([ones_col, np.sin(X_[:, ::-1] * np.pi / 2)])
        f = (1 + g)[:, None] - (1 + g)[:, None] * cp * rev
        scale = 2.0 ** np.arange(1, M + 1)
        out["F"] = f * scale

    def _calc_pareto_front(self, ref_dirs=None):
        if ref_dirs is None:
            ref_dirs = _uniform_simplex_points(self.n_obj)
        R = ref_dirs / np.linalg.norm(ref_dirs, axis=1, keepdims=True)
        scale = 2.0 ** np.arange(1, self.n_obj + 1)
        return (1 - R) * scale


class MaF5(Problem):
    """DTLZ4 mise a l'echelle -- biais alpha=100 de DTLZ4, objectifs 2^M..2^1."""

    def __init__(self, n_var, n_obj):
        super().__init__(n_var=n_var, n_obj=n_obj, xl=0.0, xu=1.0, vtype=float)

    def _evaluate(self, x, out, *args, **kwargs):
        M = self.n_obj
        n = x.shape[0]
        X_ = x[:, :M - 1] ** 100
        X_M = x[:, M - 1:]
        g = np.sum((X_M - 0.5) ** 2, axis=1)
        ones_col = np.ones((n, 1))
        cp = np.fliplr(np.cumprod(np.hstack([ones_col, np.cos(X_ * np.pi / 2)]), axis=1))
        rev = np.hstack([ones_col, np.sin(X_[:, ::-1] * np.pi / 2)])
        f = (1 + g)[:, None] * cp * rev
        scale = 2.0 ** np.arange(M, 0, -1)
        out["F"] = f * scale

    def _calc_pareto_front(self, ref_dirs=None):
        if ref_dirs is None:
            ref_dirs = _uniform_simplex_points(self.n_obj)
        R = ref_dirs / np.linalg.norm(ref_dirs, axis=1, keepdims=True)
        scale = 2.0 ** np.arange(self.n_obj, 0, -1)
        return R * scale


class MaF6(Problem):
    """DTLZ5(I=2,M) -- front degenere : une courbe a 1 parametre plongee en dimension M."""

    _I = 2  # PlatEMO fixe I=2 : exactement un angle libre quel que soit M

    def __init__(self, n_var, n_obj):
        super().__init__(n_var=n_var, n_obj=n_obj, xl=0.0, xu=1.0, vtype=float)

    def _evaluate(self, x, out, *args, **kwargs):
        M, I = self.n_obj, self._I
        n = x.shape[0]
        X_, X_M = x[:, :M - 1].copy(), x[:, M - 1:]
        g = np.sum((X_M - 0.5) ** 2, axis=1)
        if M - 1 >= I:
            temp = g[:, None]
            X_[:, I - 1:M - 1] = (1 + 2 * temp * X_[:, I - 1:M - 1]) / (2 + 2 * temp)
        ones_col = np.ones((n, 1))
        cp = np.fliplr(np.cumprod(np.hstack([ones_col, np.cos(X_ * np.pi / 2)]), axis=1))
        rev = np.hstack([ones_col, np.sin(X_[:, ::-1] * np.pi / 2)])
        out["F"] = (1 + 100 * g)[:, None] * cp * rev

    def _calc_pareto_front(self, ref_dirs=None):
        M, I = self.n_obj, self._I
        R2 = get_reference_directions("das-dennis", I, n_partitions=10000)
        R2 = R2 / np.linalg.norm(R2, axis=1, keepdims=True)
        pad = np.repeat(R2[:, :1], M - I, axis=1)
        R = np.hstack([pad, R2])
        exponents = np.concatenate([[M - I], np.arange(M - I, 2 - I - 1, -1)])
        exponents = np.maximum(exponents, 0)
        return R / (np.sqrt(2.0) ** exponents)[None, :]


class MaF7(Problem):
    """DTLZ7 -- front de Pareto disconnecte (2^(M-1) regions separees)."""

    def __init__(self, n_var, n_obj):
        super().__init__(n_var=n_var, n_obj=n_obj, xl=0.0, xu=1.0, vtype=float)

    def _evaluate(self, x, out, *args, **kwargs):
        M = self.n_obj
        n = x.shape[0]
        X_, X_M = x[:, :M - 1], x[:, M - 1:]
        g = 1 + 9 * np.mean(X_M, axis=1)
        f = np.empty((n, M))
        f[:, :M - 1] = X_
        f[:, M - 1] = (1 + g) * (M - np.sum(X_ / (1 + g)[:, None] * (1 + np.sin(3 * np.pi * X_)), axis=1))
        out["F"] = f

    _GRID_PTS = {2: 100, 3: 22}

    def _calc_pareto_front(self, ref_dirs=None):
        M = self.n_obj
        interval = np.array([0.0, 0.251412, 0.631627, 0.859401])
        median = (interval[1] - interval[0]) / (interval[3] - interval[2] + interval[1] - interval[0])
        pts = self._GRID_PTS.get(M - 1, max(2, round(10000 ** (1.0 / (M - 1)))))
        axes = [np.linspace(0.0, 1.0, pts) for _ in range(M - 1)]
        mesh = np.meshgrid(*axes, indexing="ij")
        X = np.stack([m.ravel() for m in mesh], axis=1)
        lo, hi = X <= median, X > median
        X = X.copy()
        X[lo] = X[lo] * (interval[1] - interval[0]) / median + interval[0]
        X[hi] = (X[hi] - median) * (interval[3] - interval[2]) / (1 - median) + interval[2]
        last = 2 * (M - np.sum(X / 2 * (1 + np.sin(3 * np.pi * X)), axis=1))
        return np.hstack([X, last[:, None]])


_MAF_CLASSES = {
    "MAF1": MaF1, "MAF2": MaF2, "MAF3": MaF3, "MAF4": MaF4,
    "MAF5": MaF5, "MAF6": MaF6, "MAF7": MaF7,
}

MAF_PROBLEM_NAMES = ["MaF1", "MaF2", "MaF3", "MaF4", "MaF5", "MaF6", "MaF7"]
MAF_DEGENERATE_PROBLEMS = set()  # MaF1-7 ont toutes un front analytique, quel que soit M


def get_maf_problem(name: str, n_obj: int = 4):
    """Retourne (probleme pymoo, n_generations) pour un nom MaF1..MaF7."""
    key = name.upper()
    if key not in _MAF_CLASSES:
        raise ValueError(f"Probleme inconnu '{name}'. Choix : {MAF_PROBLEM_NAMES}")
    n_var = n_obj + _MAF_K[key] - 1
    problem = _MAF_CLASSES[key](n_var=n_var, n_obj=n_obj)
    _, pop_size = get_run_config(n_obj)
    n_gen = TMAX // pop_size
    return problem, n_gen


### 6. Indicateur de qualite IGD (Inverted Generational Distance)

In [ ]:
# =============================================================================
# 5. INDICATEUR DE QUALITE -- IGD (Inverted Generational Distance)
# =============================================================================

_P_STAR_PARTITIONS = {3: 140, 4: 37}


def _get_true_front(problem):
    p = _P_STAR_PARTITIONS.get(problem.n_obj, 12)
    ref_dirs = get_reference_directions("das-dennis", problem.n_obj, n_partitions=p)
    try:
        pf = problem.pareto_front(ref_dirs=ref_dirs)
    except TypeError:
        pf = problem.pareto_front()
    if pf is None:
        raise ValueError(
            f"Impossible de recuperer le front de Pareto vrai pour {type(problem).__name__}."
        )
    return pf


def compute_igd(problem, approx_front: np.ndarray) -> float:
    """IGD entre approx_front et le vrai front de Pareto du probleme. Plus bas = mieux."""
    true_front = _get_true_front(problem)
    indicator = IGD(true_front)
    return float(indicator(approx_front))


def igd_statistics(igd_values: list) -> dict:
    """best/median/worst/mean/std d'IGD sur plusieurs runs independants."""
    arr = np.array(igd_values, dtype=float)
    return {
        "best":   float(np.min(arr)),
        "median": float(np.median(arr)),
        "worst":  float(np.max(arr)),
        "mean":   float(np.mean(arr)),
        "std":    float(np.std(arr)),
    }


### 7. QI-NSGA-III — population quantique

In [ ]:
# =============================================================================
# 6. QI-NSGA-III -- POPULATION QUANTIQUE (identique au solveur IRP du projet)
# =============================================================================

_ROTATION_TYPES = ("tanh", "tanh_soft", "linear")


class QuantumPopulation:
    """Population de chromosomes quantiques stockee comme matrice d'angles theta."""

    def __init__(
        self,
        pop_size: int,
        n_genes: int,
        xl: np.ndarray,
        xu: np.ndarray,
        rng: np.random.Generator | None = None,
        rotation_type: str = "tanh",
        noise_scale: float = 0.02,
    ) -> None:
        if rotation_type not in _ROTATION_TYPES:
            raise ValueError(f"rotation_type doit etre parmi {_ROTATION_TYPES}, recu '{rotation_type}'")
        self.pop_size      = pop_size
        self.n_genes       = n_genes
        self.xl            = np.asarray(xl, dtype=float)
        self.xu            = np.asarray(xu, dtype=float)
        self.rng           = rng if rng is not None else np.random.default_rng()
        self.rotation_type = rotation_type
        self.noise_scale   = noise_scale

        self.theta = np.full((pop_size, n_genes), np.pi / 4.0)
        self.theta += self.rng.uniform(-0.05, 0.05, (pop_size, n_genes))
        self.theta  = np.clip(self.theta, 0.0, np.pi / 2.0)

    def measure(self) -> np.ndarray:
        p     = np.cos(self.theta) ** 2
        mu    = self.xl + p * (self.xu - self.xl)
        sigma = self.noise_scale * np.abs(np.sin(2.0 * self.theta)) * (self.xu - self.xl)
        noise = self.rng.standard_normal(self.theta.shape) * sigma
        return np.clip(mu + noise, self.xl, self.xu)

    def rotate(self, guides_theta: np.ndarray, alpha: float) -> None:
        diff = guides_theta - self.theta
        if self.rotation_type == "tanh":
            self.theta += alpha * np.tanh(diff / (np.pi / 8.0))
        elif self.rotation_type == "tanh_soft":
            self.theta += alpha * np.tanh(diff / (np.pi / 4.0))
        else:  # "linear"
            self.theta += alpha * diff / (np.pi / 2.0)
        self.theta = np.clip(self.theta, 0.0, np.pi / 2.0)


### 8. QI-NSGA-III — fonctions internes generiques (tri, niching, archive)

In [ ]:
# =============================================================================
# 7. QI-NSGA-III -- FONCTIONS INTERNES GENERIQUES
#    (tri non-domine, normalisation, niching, archive, distance de foule --
#    identiques a celles utilisees par le solveur IRP du projet)
# =============================================================================

def _penalised_F(F: np.ndarray, G: np.ndarray) -> np.ndarray:
    cv         = np.maximum(G, 0.0)
    n_violated = (cv > 0).sum(axis=1, keepdims=True)
    total_cv   = cv.sum(axis=1, keepdims=True)
    penalty    = 1e9 * n_violated + 1e6 * total_cv
    return F + penalty


def _compute_nadir(F: np.ndarray, ideal: np.ndarray) -> np.ndarray:
    """Nadir via points extremes + intersection d'hyperplan (Deb & Jain 2014, IV-A)."""
    M            = F.shape[1]
    F_translated = F - ideal
    eps          = 1e-6

    extreme_idx = []
    for i in range(M):
        w    = np.full(M, eps)
        w[i] = 1.0
        extreme_idx.append(int((F_translated / w).max(axis=1).argmin()))

    A = F_translated[extreme_idx]
    try:
        a          = np.linalg.solve(A, np.ones(M))
        intercepts = 1.0 / np.where(np.abs(a) > 1e-9, a, 1e-9)
        if np.all(intercepts > 0):
            return ideal + intercepts
    except np.linalg.LinAlgError:
        pass

    return F.max(axis=0)


def _normalise_F(
    F:     np.ndarray,
    ideal: np.ndarray | None = None,
    nadir: np.ndarray | None = None,
) -> np.ndarray:
    """Si ideal/nadir sont fournis, F est normalise directement contre eux --
    run_qinsga3_generic partage alors l'ideal/nadir courant de pymoo
    (ReferenceDirectionSurvival.norm, monotone sur tout le run) au lieu de le
    recalculer a partir de la seule population de la generation courante a
    chaque appel (voir Solvers/QINSGA3/algorithm.py::_normalise_F du projet
    pour la justification complete)."""
    if ideal is None:
        ideal = F.min(axis=0)
    if nadir is None:
        nadir = _compute_nadir(F, ideal)
    denom = np.where(nadir - ideal > 1e-9, nadir - ideal, 1.0)
    return (F - ideal) / denom


def _assign_ref_dirs(F_norm: np.ndarray, ref_dirs: np.ndarray) -> np.ndarray:
    norms    = np.linalg.norm(ref_dirs, axis=1, keepdims=True)
    ref_unit = ref_dirs / np.where(norms > 1e-9, norms, 1.0)
    proj     = F_norm @ ref_unit.T
    F_sq     = (F_norm ** 2).sum(axis=1, keepdims=True)
    dist2    = np.maximum(F_sq - proj ** 2, 0.0)
    return dist2.argmin(axis=1)


def _select_guides(
    assoc: np.ndarray, pareto_idx: np.ndarray, F_norm: np.ndarray,
    ref_dirs: np.ndarray, qpop_theta: np.ndarray,
) -> np.ndarray:
    N            = len(assoc)
    F_par_n      = F_norm[pareto_idx]
    global_fb    = qpop_theta[pareto_idx[np.linalg.norm(F_par_n, axis=1).argmin()]]
    pareto_assoc = assoc[pareto_idx]

    ref_norms = np.linalg.norm(ref_dirs, axis=1, keepdims=True)
    ref_unit  = ref_dirs / np.where(ref_norms > 1e-9, ref_norms, 1.0)

    guides_theta = np.tile(global_fb, (N, 1))

    for rd in np.unique(pareto_assoc):
        same_mask = pareto_assoc == rd
        same_idx  = pareto_idx[same_mask]

        if len(same_idx) == 1:
            best_theta = qpop_theta[same_idx[0]]
        else:
            F_same  = F_norm[same_idx]
            proj    = F_same @ ref_unit[rd]
            d_perp2 = np.maximum((F_same ** 2).sum(axis=1) - proj ** 2, 0.0)
            best_theta = qpop_theta[same_idx[d_perp2.argmin()]]

        guides_theta[assoc == rd] = best_theta

    return guides_theta


def _supplement_from_archive(
    guides_theta: np.ndarray, assoc: np.ndarray, pareto_assoc: np.ndarray,
    arch_theta: np.ndarray, arch_F_norm: np.ndarray, ref_dirs: np.ndarray,
) -> np.ndarray:
    arch_assoc = _assign_ref_dirs(arch_F_norm, ref_dirs)
    covered    = set(pareto_assoc.tolist())

    ref_norms = np.linalg.norm(ref_dirs, axis=1, keepdims=True)
    ref_unit  = ref_dirs / np.where(ref_norms > 1e-9, ref_norms, 1.0)

    pop_rds           = np.unique(assoc)
    uncovered_pop_rds = pop_rds[~np.isin(pop_rds, list(covered))]

    for rd in uncovered_pop_rds:
        in_niche = np.where(arch_assoc == rd)[0]
        if len(in_niche) == 0:
            continue
        if len(in_niche) == 1:
            best_theta = arch_theta[in_niche[0]]
        else:
            F_cand  = arch_F_norm[in_niche]
            proj    = F_cand @ ref_unit[rd]
            d_perp2 = np.maximum((F_cand ** 2).sum(axis=1) - proj ** 2, 0.0)
            best_theta = arch_theta[in_niche[d_perp2.argmin()]]

        guides_theta[assoc == rd] = best_theta

    return guides_theta


def _migrate(
    qpop: QuantumPopulation, arch_theta: np.ndarray, arch_F_norm: np.ndarray,
    assoc: np.ndarray, ref_dirs: np.ndarray, rng: np.random.Generator,
    n_migrate: int = 10,
) -> None:
    arch_assoc = _assign_ref_dirs(arch_F_norm, ref_dirs)

    targets = rng.choice(qpop.pop_size, size=min(n_migrate, qpop.pop_size), replace=False)
    for idx in targets:
        rd       = assoc[idx]
        in_niche = np.where(arch_assoc == rd)[0]
        if len(in_niche) > 0:
            r_norm  = ref_dirs[rd] / max(np.linalg.norm(ref_dirs[rd]), 1e-9)
            F_cand  = arch_F_norm[in_niche]
            proj    = F_cand @ r_norm
            d_perp2 = np.maximum((F_cand ** 2).sum(axis=1) - proj ** 2, 0.0)
            qpop.theta[idx] = arch_theta[in_niche[d_perp2.argmin()]]
        else:
            qpop.theta[idx] = arch_theta[np.linalg.norm(arch_F_norm, axis=1).argmin()]

    qpop.theta = np.clip(qpop.theta, 0.0, np.pi / 2.0)


def _crowding_distance(F: np.ndarray) -> np.ndarray:
    N, M = F.shape
    cd   = np.zeros(N)
    for m in range(M):
        order        = np.argsort(F[:, m])
        f_min, f_max = F[order[0], m], F[order[-1], m]
        cd[order[0]]  = np.inf
        cd[order[-1]] = np.inf
        span = f_max - f_min if f_max - f_min > 1e-9 else 1.0
        cd[order[1:-1]] += (F[order[2:], m] - F[order[:-2], m]) / span
    return cd


def _crowding_trim(X: np.ndarray, F: np.ndarray, theta: np.ndarray, max_size: int):
    if len(X) <= max_size:
        return X, F, theta
    keep = np.argsort(_crowding_distance(_normalise_F(F)))[-max_size:]
    return X[keep], F[keep], theta[keep]


def _archive_update(
    new_X: np.ndarray, new_F: np.ndarray, new_G: np.ndarray, new_theta: np.ndarray,
    arch_X: list, arch_F: list, arch_theta: list, max_size: int = 500,
) -> None:
    feasible = np.maximum(new_G, 0.0).sum(axis=1) == 0
    feas_idx = np.where(feasible)[0]
    if len(feas_idx) == 0:
        return

    cand_X     = new_X[feas_idx]
    cand_F     = new_F[feas_idx]
    cand_theta = new_theta[feas_idx]

    n_arch = len(arch_F)
    arr    = np.array(arch_F) if n_arch > 0 else None
    alive  = np.ones(n_arch, dtype=bool)

    add_X, add_F, add_theta = [], [], []

    for c in range(len(cand_F)):
        f = cand_F[c]

        if arr is not None:
            arr_live = arr[alive]
            if len(arr_live):
                if ((arr_live <= f).all(axis=1) & (arr_live < f).any(axis=1)).any():
                    continue
                if (np.abs(arr_live - f).max(axis=1) < 1e-6).any():
                    continue
                live_idx = np.where(alive)[0]
                dom = (f <= arr_live).all(axis=1) & (f < arr_live).any(axis=1)
                alive[live_idx[dom]] = False

        if add_F:
            add_arr = np.array(add_F)
            if (np.abs(add_arr - f).max(axis=1) < 1e-6).any():
                continue

        add_X.append(cand_X[c].copy())
        add_F.append(f.copy())
        add_theta.append(cand_theta[c].copy())

    if n_arch > 0 and not alive.all():
        keep = np.where(alive)[0].tolist()
        arch_X[:]     = [arch_X[i]     for i in keep]
        arch_F[:]     = [arch_F[i]     for i in keep]
        arch_theta[:] = [arch_theta[i] for i in keep]

    arch_X.extend(add_X)
    arch_F.extend(add_F)
    arch_theta.extend(add_theta)

    if len(arch_F) > max_size:
        arr_X, arr_F, arr_theta = _crowding_trim(
            np.array(arch_X), np.array(arch_F), np.array(arch_theta), max_size,
        )
        arch_X[:]     = list(arr_X)
        arch_F[:]     = list(arr_F)
        arch_theta[:] = list(arr_theta)


def _encode_theta(X: np.ndarray, xl: np.ndarray, xu: np.ndarray) -> np.ndarray:
    p = np.clip((X - xl) / np.where(xu - xl > 1e-12, xu - xl, 1.0), 0.0, 1.0)
    return np.clip(np.arccos(np.sqrt(p)), 0.0, np.pi / 2.0)


### 9. QI-NSGA-III — boucle generique (n'importe quel probleme pymoo)

In [ ]:
# =============================================================================
# 8. QI-NSGA-III -- BOUCLE GENERIQUE (n'importe quel probleme pymoo)
#
#    Contrairement au solveur IRP (evaluation couteuse -> multiprocessing),
#    DTLZ/MaF sont des fonctions analytiques bon marche : toute la population
#    est evaluee en un seul appel vectorise `problem.evaluate(X, ...)` par
#    generation, sans pool de process.
# =============================================================================

def run_qinsga3_generic(
    problem, ref_dirs: np.ndarray, pop_size: int, max_gen: int,
    alpha_max: float, alpha_min: float, p_cross: float, eta_cross: float,
    p_mut: float, eta_mut: float, migration_period: int, n_migrate: int,
    seed: int, rotation_type: str = "tanh", noise_scale: float = 0.02,
    escape_prob: float = 0.0,
) -> np.ndarray:
    """Execute une instance de QI-NSGA-III sur un probleme pymoo. Retourne F
    du front de Pareto final (objectifs reels).

    escape_prob : probabilite par gene de reinitialiser theta a une valeur
    Uniform(0, pi/2) apres l'etape de variation en X-space (0 = desactive).
    Corrige un effondrement de population sur DTLZ4/MaF5 (leur biais alpha=100
    ecrase presque toute la population au meme point des la generation 0) --
    voir Validation/Benchmarking/algorithms/qinsga3/core.py du projet pour le
    detail de l'ablation qui a valide ce mecanisme.
    """
    rng = np.random.default_rng(seed)

    xl = np.asarray(problem.xl, dtype=float)
    xu = np.asarray(problem.xu, dtype=float)
    n_genes = problem.n_var

    qpop     = QuantumPopulation(
        pop_size, n_genes, xl, xu, rng=rng,
        rotation_type=rotation_type, noise_scale=noise_scale,
    )
    sorter   = NonDominatedSorting()
    survival = ReferenceDirectionSurvival(ref_dirs)
    sbx_op   = SBX(prob=p_cross, eta=eta_cross)
    pm_op    = PM(prob=p_mut,    eta=eta_mut)

    arch_X: list[np.ndarray] = []
    arch_F: list[np.ndarray] = []
    arch_theta: list[np.ndarray] = []
    _MAX_ARCHIVE = 500

    def _eval_batch(X: np.ndarray):
        F, G = problem.evaluate(X, return_values_of=["F", "G"])
        if G is None or (hasattr(G, "size") and G.size == 0):
            G = np.zeros((X.shape[0], 0))
        return np.asarray(F), np.asarray(G)

    for gen in range(max_gen):
        theta_parent = qpop.theta.copy()
        X_parent     = qpop.measure()
        F_parent, G_parent = _eval_batch(X_parent)
        F_pen_parent = _penalised_F(F_parent, G_parent)

        pareto_idx = sorter.do(F_pen_parent)[0]
        _archive_update(
            X_parent[pareto_idx], F_parent[pareto_idx], G_parent[pareto_idx],
            theta_parent[pareto_idx], arch_X, arch_F, arch_theta, _MAX_ARCHIVE,
        )

        # Partage le MEME ideal/nadir que l'etape de survie elitiste ci-dessous
        # (survival.norm de pymoo, monotone sur tout le run) pour la selection
        # des guides -- repli sur l'ancien calcul a la generation 0 (avant que
        # survival.do() ait tourne une fois).
        if survival.norm.nadir_point is None:
            F_norm = _normalise_F(F_pen_parent)
        else:
            F_norm = _normalise_F(F_pen_parent, survival.norm.ideal_point, survival.norm.nadir_point)
        assoc  = _assign_ref_dirs(F_norm, ref_dirs)
        guides_theta = _select_guides(assoc, pareto_idx, F_norm, ref_dirs, qpop.theta)

        arch_theta_arr = None
        arch_F_norm    = None
        if len(arch_X) >= 4:
            arch_theta_arr = np.array(arch_theta)
            if survival.norm.nadir_point is None:
                arch_F_norm = _normalise_F(np.array(arch_F))
            else:
                arch_F_norm = _normalise_F(np.array(arch_F), survival.norm.ideal_point, survival.norm.nadir_point)
            pareto_assoc   = assoc[pareto_idx]
            guides_theta   = _supplement_from_archive(
                guides_theta, assoc, pareto_assoc,
                arch_theta_arr, arch_F_norm, ref_dirs,
            )

        alpha = alpha_min + (alpha_max - alpha_min) * (1.0 - gen / max_gen)

        qpop.rotate(guides_theta, alpha)
        X_rotated = qpop.measure()

        if p_cross > 0.0:
            idx     = rng.permutation(pop_size)
            n_pairs = pop_size // 2
            pairs   = idx[: n_pairs * 2].reshape(n_pairs, 2)
            X_pairs = np.transpose(X_rotated[pairs], (1, 0, 2))
            Q       = sbx_op._do(problem, X_pairs, random_state=rng)
            X_rotated[pairs[:, 0]] = Q[0]
            X_rotated[pairs[:, 1]] = Q[1]
        X_varied = np.clip(pm_op._do(problem, X_rotated, random_state=rng), xl, xu)

        theta_offspring = _encode_theta(X_varied, xl, xu)

        if escape_prob > 0.0:
            escape_mask = rng.random(theta_offspring.shape) < escape_prob
            if escape_mask.any():
                theta_offspring[escape_mask] = rng.uniform(0.0, np.pi / 2.0, int(escape_mask.sum()))

        qpop.theta      = theta_offspring
        X_offspring     = qpop.measure()
        F_offspring, G_offspring = _eval_batch(X_offspring)
        F_pen_offspring = _penalised_F(F_offspring, G_offspring)

        off_pareto_idx = sorter.do(F_pen_offspring)[0]
        _archive_update(
            X_offspring[off_pareto_idx], F_offspring[off_pareto_idx], G_offspring[off_pareto_idx],
            theta_offspring[off_pareto_idx], arch_X, arch_F, arch_theta, _MAX_ARCHIVE,
        )

        # Selection elitiste : merge parent+enfants, garde les pop_size
        # meilleurs via la survie NSGA-III native de pymoo (feasibility-first
        # -- voir le meme correctif applique au solveur IRP de ce projet ;
        # DTLZ/MaF n'ayant aucune contrainte, ceci est equivalent a l'ancien
        # code mais reste la version la plus correcte/generale).
        theta_pool  = np.vstack([theta_parent, theta_offspring])
        F_true_pool = np.vstack([F_parent, F_offspring])
        G_pool      = np.vstack([G_parent, G_offspring])
        merged_pop  = Population.new(X=theta_pool, F=F_true_pool, G=G_pool)
        survived    = survival.do(problem, merged_pop, n_survive=pop_size, random_state=rng)
        qpop.theta  = np.clip(np.asarray(survived.get("X"), dtype=float), 0.0, np.pi / 2.0)

        if (arch_F_norm is not None
                and migration_period > 0
                and gen % migration_period == 0):
            _migrate(
                qpop, arch_theta_arr, arch_F_norm,
                assoc, ref_dirs, rng, n_migrate=n_migrate,
            )

    X_final = qpop.measure()
    F_final, G_final = _eval_batch(X_final)
    F_pen_final = _penalised_F(F_final, G_final)
    final_pareto_idx = sorter.do(F_pen_final)[0]
    _archive_update(
        X_final[final_pareto_idx], F_final[final_pareto_idx],
        G_final[final_pareto_idx], qpop.theta[final_pareto_idx],
        arch_X, arch_F, arch_theta, _MAX_ARCHIVE,
    )

    if arch_X:
        _, arch_F_arr, _ = _crowding_trim(
            np.array(arch_X), np.array(arch_F), np.array(arch_theta), pop_size,
        )
        return arch_F_arr

    return F_final[final_pareto_idx]


### 10. NSGA-III — lanceur benchmark

In [ ]:
# =============================================================================
# 9. NSGA-III -- LANCEUR BENCHMARK (pymoo natif)
#    Cui et al. 2025, Table 2 : SBX eta=20 pc=1.0, PM eta=20 pm=1/D.
# =============================================================================

def nsga3_run_single(problem, n_gen: int, seed: int) -> np.ndarray:
    """Execute un run NSGA-III, retourne les objectifs non-domines (n_sol, n_obj)."""
    np.random.seed(seed)
    ref_dirs, pop_size = get_run_config(problem.n_obj)

    algorithm = NSGA3(
        pop_size=pop_size,
        ref_dirs=ref_dirs,
        sampling=FloatRandomSampling(),
        crossover=SBX(prob=1.0, eta=20),
        mutation=PM(prob=1.0 / problem.n_var, eta=20),
    )

    result = minimize(
        problem, algorithm, get_termination("n_gen", n_gen),
        seed=seed, verbose=False,
    )

    if result.F is None or len(result.F) == 0:
        raise RuntimeError(
            f"NSGA-III n'a retourne aucune solution pour {type(problem).__name__} "
            f"(n_obj={problem.n_obj}) avec seed={seed}."
        )
    return result.F


def nsga3_run_experiment(problem_name: str, problem, n_gen: int, n_runs: int = 30) -> list:
    """Execute NSGA-III n_runs fois (seeds distinctes). Retourne une liste de fronts F."""
    if n_runs > len(SEEDS):
        raise ValueError(f"n_runs={n_runs} depasse le nombre de seeds disponibles ({len(SEEDS)})")

    fronts = []
    for i in range(n_runs):
        seed = SEEDS[i]
        print(f"  [{problem_name}] Run {i+1:02d}/{n_runs}  seed={seed}", flush=True)
        front = nsga3_run_single(problem, n_gen, seed)
        fronts.append(front)
        print(f"  [{problem_name}] Run {i+1:02d} termine - taille front: {len(front)}", flush=True)
    return fronts


### 11. QI-NSGA-III — lanceur benchmark

In [ ]:
# =============================================================================
# 10. QI-NSGA-III -- LANCEUR BENCHMARK
#     Hyperparametres specifiques DTLZ/MaF (re-tunes par ablation dans ce
#     projet ; voir Validation/Benchmarking/algorithms/qinsga3/runner.py) :
#     alignes sur NSGA-III pour SBX/PM (pc=1.0, eta=20, pm=1/D), le reste
#     (rotation, migration, escape_prob) reste specifique a l'encodage
#     quantique.
# =============================================================================

QINSGA3_ALPHA_MAX = 0.10 * np.pi
QINSGA3_ALPHA_MIN = 0.001 * np.pi
QINSGA3_P_CROSS = 1.0
QINSGA3_ETA_CROSS = 20.0
QINSGA3_ETA_MUT = 20.0
QINSGA3_MIGRATION_PERIOD = 5
QINSGA3_N_MIGRATE = 20
QINSGA3_ROTATION_TYPE = "tanh"
QINSGA3_NOISE_SCALE = 0.0   # pas d'arrondi entier a proteger ici (contrairement a l'IRP)
QINSGA3_ESCAPE_PROB_FACTOR = 2.0 * 0.15  # -> escape_prob = FACTOR / n_var


def qinsga3_run_single(problem, n_gen: int, seed: int) -> np.ndarray:
    """Execute un run QI-NSGA-III, retourne les objectifs du front final."""
    ref_dirs, pop_size = get_run_config(problem.n_obj)
    p_mut = 1.0 / problem.n_var
    escape_prob = QINSGA3_ESCAPE_PROB_FACTOR / problem.n_var

    return run_qinsga3_generic(
        problem, ref_dirs, pop_size, max_gen=n_gen,
        alpha_max=QINSGA3_ALPHA_MAX, alpha_min=QINSGA3_ALPHA_MIN,
        p_cross=QINSGA3_P_CROSS, eta_cross=QINSGA3_ETA_CROSS,
        p_mut=p_mut, eta_mut=QINSGA3_ETA_MUT,
        migration_period=QINSGA3_MIGRATION_PERIOD, n_migrate=QINSGA3_N_MIGRATE,
        seed=seed, rotation_type=QINSGA3_ROTATION_TYPE, noise_scale=QINSGA3_NOISE_SCALE,
        escape_prob=escape_prob,
    )


def qinsga3_run_experiment(problem_name: str, problem, n_gen: int, n_runs: int = 30) -> list:
    """Execute QI-NSGA-III n_runs fois (memes seeds que NSGA-III). Retourne une liste de fronts F."""
    if n_runs > len(SEEDS):
        raise ValueError(f"n_runs={n_runs} depasse le nombre de seeds disponibles ({len(SEEDS)})")

    fronts = []
    for i in range(n_runs):
        seed = SEEDS[i]
        print(f"  [{problem_name}] Run {i+1:02d}/{n_runs}  seed={seed}", flush=True)
        front = qinsga3_run_single(problem, n_gen, seed)
        fronts.append(front)
        print(f"  [{problem_name}] Run {i+1:02d} termine - taille front: {len(front)}", flush=True)
    return fronts


### 12. Moteur de validation (execute un algo sur toute une suite)

In [ ]:
# =============================================================================
# 11. MOTEUR DE VALIDATION -- execute un algo sur toute une suite de problemes
# =============================================================================

class _ProblemSuite:
    """Petit conteneur imitant l'interface attendue par validate() :
    PROBLEM_NAMES, DEGENERATE_PROBLEMS, get_problem(name, n_obj)."""

    def __init__(self, problem_names, degenerate_problems, get_problem_fn):
        self.PROBLEM_NAMES = problem_names
        self.DEGENERATE_PROBLEMS = degenerate_problems
        self.get_problem = get_problem_fn


DTLZ_SUITE = _ProblemSuite(DTLZ_PROBLEM_NAMES, DTLZ_DEGENERATE_PROBLEMS, get_dtlz_problem)
MAF_SUITE  = _ProblemSuite(MAF_PROBLEM_NAMES, MAF_DEGENERATE_PROBLEMS, get_maf_problem)


def _save_run_csv(results_dir: str, problem_name: str, igd_values: list, n_obj: int) -> str:
    os.makedirs(results_dir, exist_ok=True)
    path = os.path.join(results_dir, f"igd_{problem_name}_M{n_obj}.csv")
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["run", "seed", "igd"])
        for i, igd in enumerate(igd_values):
            writer.writerow([i + 1, SEEDS[i], f"{igd:.6f}"])
    return path


def _save_summary_csv(results_dir: str, summary_rows: list, n_obj: int) -> str:
    os.makedirs(results_dir, exist_ok=True)
    path = os.path.join(results_dir, f"summary_M{n_obj}.csv")
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Problem", "IGD_Best", "IGD_Median", "IGD_Worst", "IGD_Mean", "IGD_Std"])
        for row in summary_rows:
            writer.writerow([
                row["problem"], f"{row['best']:.6f}", f"{row['median']:.6f}",
                f"{row['worst']:.6f}", f"{row['mean']:.6f}", f"{row['std']:.6f}",
            ])
    return path


def _print_table(suite_label: str, summary_rows: list, n_obj: int, algorithm_label: str):
    header = f"{'Problem':<10} {'IGD Best':>12} {'IGD Median':>12} {'IGD Worst':>12} {'IGD Mean':>12} {'IGD Std':>12}"
    sep = "-" * len(header)
    print("\n" + sep)
    print(f"  {algorithm_label} sur {suite_label} ({n_obj} objectifs) - protocole Cui et al. 2025")
    print(sep)
    print(header)
    print(sep)
    for row in summary_rows:
        print(
            f"{row['problem']:<10} {row['best']:>12.6f} {row['median']:>12.6f} "
            f"{row['worst']:>12.6f} {row['mean']:>12.6f} {row['std']:>12.6f}"
        )
    print(sep + "\n")


def validate(problems_module, results_dir: str, suite_label: str, n_runs: int = 30, n_obj: int = 4,
             run_experiment_fn=nsga3_run_experiment, algorithm_label: str = "NSGA-III"):
    """Execute un algo sur chaque probleme de problems_module.PROBLEM_NAMES.

    Retourne une liste de dicts resume : problem, best, median, worst, mean, std.
    """
    summary_rows = []

    for name in problems_module.PROBLEM_NAMES:
        print(f"\n{'='*50}")
        print(f"Probleme: {name}  ({n_runs} runs, {n_obj} objectifs)")
        print(f"{'='*50}")

        if name in problems_module.DEGENERATE_PROBLEMS and n_obj != 3:
            print(
                f"  {name} ignore a M={n_obj} : pas de front de Pareto analytique "
                f"disponible pour ce probleme au-dela de M=3."
            )
            continue

        problem, n_gen = problems_module.get_problem(name, n_obj=n_obj)
        print(f"  n_var={problem.n_var}, n_obj={problem.n_obj}, n_gen={n_gen}")

        t0 = time.time()
        fronts = run_experiment_fn(name, problem, n_gen, n_runs)
        elapsed = time.time() - t0
        print(f"  {name}: {n_runs} runs termines en {elapsed:.1f}s", flush=True)

        igd_values = [compute_igd(problem, front) for front in fronts]
        stats = igd_statistics(igd_values)

        csv_path = _save_run_csv(results_dir, name, igd_values, n_obj)
        print(f"  IGD par run sauvegarde -> {csv_path}")

        summary_rows.append({
            "problem": name, "best": stats["best"], "median": stats["median"],
            "worst": stats["worst"], "mean": stats["mean"], "std": stats["std"],
        })

    summary_path = _save_summary_csv(results_dir, summary_rows, n_obj)
    print(f"\nResume CSV -> {summary_path}")

    _print_table(suite_label, summary_rows, n_obj, algorithm_label)
    return summary_rows


### 13. Point d'entree (comparatif + telechargement Colab)

In [ ]:
# =============================================================================
# 12. POINT D'ENTREE -- lance NSGA-III et QI-NSGA-III sur DTLZ1-7 et MaF1-7,
#     compare, sauvegarde tout.
# =============================================================================

_SUITES = {"DTLZ": DTLZ_SUITE, "MaF": MAF_SUITE}
_ALGOS = {
    "NSGA-III":    nsga3_run_experiment,
    "QI-NSGA-III": qinsga3_run_experiment,
}


def _print_comparison_table(suite_name, n_obj, rows_by_algo):
    """Tableau IGD moyen NSGA-III vs QI-NSGA-III, cote a cote, par probleme."""
    problems = [r["problem"] for r in rows_by_algo["NSGA-III"]]
    by_problem = {
        algo: {r["problem"]: r for r in rows}
        for algo, rows in rows_by_algo.items()
    }

    header = f"{'Probleme':<10}{'NSGA-III (mean IGD)':>22}{'QI-NSGA-III (mean IGD)':>24}{'Gagnant':>14}"
    print("\n" + "=" * len(header))
    print(f"COMPARATIF {suite_name} (M={n_obj}) -- IGD moyen, plus bas = mieux")
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    for p in problems:
        n3 = by_problem["NSGA-III"].get(p)
        qn3 = by_problem["QI-NSGA-III"].get(p)
        if n3 is None or qn3 is None:
            continue
        winner = "NSGA-III" if n3["mean"] < qn3["mean"] else "QI-NSGA-III"
        print(f"{p:<10}{n3['mean']:>22.6f}{qn3['mean']:>24.6f}{winner:>14}")
    print("=" * len(header))


def run_all_and_compare(
    suites=("DTLZ", "MaF"), n_obj_list=(3, 4), n_runs=5, results_dir="benchmark_results",
):
    """Lance NSGA-III et QI-NSGA-III sur toutes les suites/n_obj demandes.

    Retourne un dict {(suite, n_obj): {"NSGA-III": rows, "QI-NSGA-III": rows}}.
    """
    all_results = {}

    for suite_name in suites:
        suite = _SUITES[suite_name]
        for n_obj in n_obj_list:
            rows_by_algo = {}
            for algo_name, run_fn in _ALGOS.items():
                algo_dir = os.path.join(results_dir, suite_name.lower(), algo_name.lower().replace("-", "_"))
                rows = validate(
                    suite, algo_dir, suite_label=f"{suite_name}1-7",
                    n_runs=n_runs, n_obj=n_obj,
                    run_experiment_fn=run_fn, algorithm_label=algo_name,
                )
                rows_by_algo[algo_name] = rows
            all_results[(suite_name, n_obj)] = rows_by_algo
            _print_comparison_table(suite_name, n_obj, rows_by_algo)

    return all_results


def _running_in_colab() -> bool:
    return "google.colab" in sys.modules


## Execution

`N_RUNS` = nombre de runs independants par probleme (5 = rapide, 30 = protocole
complet de l'etude). `N_OBJ_LIST` = nombre(s) d'objectifs a tester (3 et/ou 4
— DTLZ5/6/7 ne sont valides qu'a M=3, pas de front analytique au-dela).
`SUITES` = quelles suites tester.

In [ ]:
# ============================================================
# PARAMETRES — modifiez ici si besoin
# ============================================================
SUITES      = ("DTLZ", "MaF")   # suites a valider
N_OBJ_LIST  = (3, 4)            # nombre(s) d'objectifs
N_RUNS      = 5                 # runs independants par probleme (etude complete: 30)
RESULTS_DIR = "/content/benchmark_results"

t_start = time.time()
all_results = run_all_and_compare(
    suites=SUITES, n_obj_list=N_OBJ_LIST, n_runs=N_RUNS, results_dir=RESULTS_DIR,
)
print(f"\nTemps total : {time.time() - t_start:.1f}s")
print(f"Resultats dans : {os.path.abspath(RESULTS_DIR)}")

zip_path = shutil.make_archive(RESULTS_DIR, "zip", RESULTS_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("(Hors Colab) Archive des resultats :", zip_path)
